1. Load Model and Dependencies 

In [4]:
%pip install -q transformers accelerate bitsandbytes datasets pandas tqdm sae-lens huggingface_hub

In [ ]:
import os
import torch
import random
import torch.nn as nn
import pandas as pd 
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sae_lens import SAE
from huggingface_hub import login

In [ ]:
from google.colab import userdata
userdata.get('HF_TOKEN')

hf_token = userdata.get('HF_TOKEN')

os.environ["HF_TOKEN"] = hf_token

login(token=hf_token)

2. Configure the model, SAE, and tasks benchmark for evaluation 

In [ ]:
model_id = "google/gemma-2-2b"
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto"
)

In [ ]:
sae, cfg_dict, sparsity = SAE.from_pretrained(
    release="gemma-scope-2b-pt-res-canonical",
    sae_id="layer_20/width_16k/canonical",
    device="cuda" if torch.cuda.is_available() else "cpu"
)

In [ ]:
def hook_residual_stream(module, input, output):
    global residual_stream_cache
    # In Hugging Face Gemma, the layer output is a tuple; index 0 is the hidden states tensor
    if isinstance(output, tuple):
        residual_stream_cache = output[0].detach()
    else:
        residual_stream_cache = output.detach()

# Register hook to layer 20 of the HF architecture
hook_handle = model.model.layers[20].register_forward_hook(hook_residual_stream)

We need to store the size of the residual stream and feature selection from our SAE for our models architecture. 

In [ ]:
total_features = cfg_dict['d_sae']
model_dimensions = cfg_dict['d_in']

print(f"Total features available: {total_features}")

LegalBench is consisted of 162 tasks, spreading across 6 categories. Issue spotting is one of these categories, and consists entirley of binary yes / no question to test a agents ability on identifying legal issues. 

Each one of these tasks is comprised of samples, and we will need to download these samples to use a training and testing data for our agent. Issue spotting has 17 different categories, most derived from the learned hands dataset. All together they have 11,512 test samples to pull from. 

In [ ]:
ISSUE_SPOTTING_TASK = ["corporate_lobbying", "learned_hands_benefits", "learned_hands_business", "learned_hands_consumer", "learned_hands_courts", "learned_hands_courts", "learned_hands_crime", "learned_hands_divorce", "learned_hands_domestic_violence", "learned_hands_employment", "learned_hands_education", "learned_hands_estates", "learned_hands_family", "learned_hands_health", "learned_hands_housing", "learned_hands_immigration", "learned_hands_torts", "learned_hands_traffic" ]

Here we load our benchmark dataset, LegalBench. LegalBench train splits only have 6 tasks in each, so instead of using that as our training data we will create our own training, validation, and test splits from the much larger test datasets from LegalBench.

In [ ]:
datasets = {}

for task in ISSUE_SPOTTING_TASK : 

    # load_dataset is the datasets library we imported at the beginning
    new_dataset = load_dataset("nguha/legalbench", task, split="test")
    datasets[task] = new_dataset



Next, we need to "shuffle" our dataset so we can sample the train test and validation segments later. It would be inefficent to load the dataset twice and there are benefits in evaluation for using a map, so we can create indices to relate to tasks and store them in a list instead. The entries of the list are in form (task name, index in task dataset).

In [ ]:
global_indices = []
for task_name, dataset in datasets.items():
    for idx in range(len(dataset)):
        global_indices.append((task_name, idx))

# shuffle the indices
random.seed(42)
random.shuffle(global_indices)


total_samples = len(global_indices)
train_end = int(total_samples * 0.6)
val_end = train_end + int(total_samples * 0.2)

3. The actor agent

In [ ]:
class Actor(nn.Module) :
    def __init__(self, d_sae, d_features) :
        super().__init__()

        # Architecture of the actor network
        self.net = nn.Sequential(
            nn.Linear(d_sae, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, d_features)
        )

    def forward(self, x):
        return self.net(x)  

4. The critic agent

In [ ]:
class Critic(nn.Module) : 
    def __init__(self, d_sae) :
        super().__init__()

        # Architecture of the actor network
        self.net = nn.Sequential(
            nn.Linear(d_sae, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 1) # one output for the value function 
        )
        
    def forward(self, x):
        return self.net(x)

Below I define the value function V(s) used in PPO, given a critic network and residual stream.

In [ ]:
def V(critic_agent, residual_stream) : 
    # compute the value function for a given residual stream 
    # residual_stream is a tensor of shape (batch_size, d_sae)
    return critic_agent(residual_stream)

5. Adaptive Feature Masking

In [ ]:
feature_mask = torch.zeros(total_features, dtype=torch.bool)

6. Training the agent

Hyperparameters

In [ ]:
training_episodes = 1000


Below we define our networks for our actor ( the policy we're learning ) and the critic ( our value function ).

In [ ]:
actor = Actor(d_sae=total_features, d_features=model_dimensions)
critic = Critic(d_sae=total_features)

Training Loop

In [ ]:
for episode in range(training_episodes) : 
    # when we train, a "episode" is a single question / pass through the model 

    #randomly select a sample from the train set of global indices 
    random_task, random_idx = random.choice(global_indices[:train_end])

    # access the sample for our training point 
    train_sample = datasets[random_task][random_idx]

    text = train_sample['text']
    target = train_sample['label']

    inputs = tokenizer(text, return_tensors="pt").to("cuda")

    # Run forward pass to trigger the hook and cache the residual stream
    with torch.no_grad():
        _ = model(**inputs)

    res_stream = residual_stream_cache[0]

    # Pass the residual stream tensor through the SAE encoder to get latent feature activations
        # Shape becomes (seq_len, sae_dictionary_width) where width is 16,384 features
    with torch.no_grad():
        feature_activations = sae.encode(res_stream)     

    # Isolate the final token slot (position t) where the next token is predicted
    x_t = res_stream[:, -1:, :] # Shape: [batch_size, 1, hidden_dim]         

    print(x_t)
    print(res_stream)
    print(feature_activations)                              
    